# Experimentos Comparativos — SentimentMLP

Compara diferentes configurações de hiperparâmetros e analisa o impacto no desempenho do modelo.

In [ ]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("Diretório de trabalho:", os.getcwd())

## 1. Carregamento e pré-processamento dos dados

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_data
from src.preprocessing.transform import preprocess_dataset, vectorize_texts, normalize_features
from src.training.train import split_dataset, to_tensors, create_dataloader, run_training
from src.models.model import build_model, predict
from src.evaluation.metrics import evaluate_model
from src.utils.config import DATA_PATH, TEXT_COLUMN, LABEL_COLUMN, TEST_SIZE, MAX_FEATURES

df = load_data(DATA_PATH)
df = preprocess_dataset(df)

x_train, x_test, y_train, y_test = split_dataset(df, TEXT_COLUMN, LABEL_COLUMN, TEST_SIZE)

vectorizer, X_train_np = vectorize_texts(x_train, MAX_FEATURES)
X_test_np = vectorizer.transform(x_test).toarray()
X_train_np = normalize_features(X_train_np)
X_test_np  = normalize_features(X_test_np)

X_train_t, y_train_t = to_tensors(X_train_np, y_train.to_numpy())
X_test_t,  y_test_t  = to_tensors(X_test_np,  y_test.to_numpy())

print(f"Train: {X_train_t.shape} | Test: {X_test_t.shape}")

## 2. Configurações experimentais

Testamos 4 configurações variando `hidden_dim`, `learning_rate` e `epochs`.

In [ ]:
configs = [
    {"name": "Baseline",    "hidden_dim": 256, "lr": 1e-3, "epochs": 10},
    {"name": "Maior rede",  "hidden_dim": 512, "lr": 1e-3, "epochs": 10},
    {"name": "LR menor",    "hidden_dim": 256, "lr": 1e-4, "epochs": 10},
    {"name": "Mais épocas", "hidden_dim": 256, "lr": 1e-3, "epochs": 20},
]

BATCH_SIZE = 64
OUTPUT_DIM = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Executando {len(configs)} experimentos...")

## 3. Treinamento de cada configuração

In [ ]:
results = []

train_loader = create_dataloader(X_train_t, y_train_t, BATCH_SIZE)
val_loader   = create_dataloader(X_test_t,  y_test_t,  BATCH_SIZE, shuffle=False)

for cfg in configs:
    print(f"\n{'='*50}")
    print(f"Config: {cfg['name']} | hidden={cfg['hidden_dim']} | lr={cfg['lr']} | epochs={cfg['epochs']}")
    print('='*50)

    model = build_model(MAX_FEATURES, cfg["hidden_dim"], OUTPUT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    criterion = nn.BCEWithLogitsLoss()

    model = run_training(
        model, train_loader, val_loader,
        optimizer, criterion, cfg["epochs"], device
    )

    y_pred = predict(model, X_test_t, device)
    metrics = evaluate_model(y_test.to_numpy(), y_pred)

    results.append({
        "Config":     cfg["name"],
        "hidden_dim": cfg["hidden_dim"],
        "lr":         cfg["lr"],
        "epochs":     cfg["epochs"],
        **{k: round(v, 4) for k, v in metrics.items()},
    })
    print(f"  Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1_score']:.4f}")

## 4. Tabela comparativa de resultados

In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results.set_index("Config")
df_results

## 5. Gráfico comparativo — Accuracy e F1

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(df_results))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], df_results["accuracy"], width, label="Accuracy", color="steelblue")
bars2 = ax.bar([i + width/2 for i in x], df_results["f1_score"],  width, label="F1-score",  color="darkorange")

ax.set_xticks(list(x))
ax.set_xticklabels(df_results.index, rotation=15, ha="right")
ax.set_ylim(0.85, 1.0)
ax.set_ylabel("Score")
ax.set_title("Comparação de Configurações — SentimentMLP")
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("results/figures/experiment_comparison.png", dpi=150)
plt.show()
print("Gráfico salvo em results/figures/experiment_comparison.png")

## 6. Análise técnica

Observe na tabela e no gráfico:
- **Baseline** (hidden=256, lr=1e-3, 10 épocas): configuração de referência.
- **Maior rede** (hidden=512): mais parâmetros podem melhorar a capacidade de representação, mas podem causar overfitting em datasets pequenos.
- **LR menor** (lr=1e-4): convergência mais lenta; pode precisar de mais épocas para atingir o mesmo desempenho.
- **Mais épocas** (20 épocas): permite que o modelo treine por mais tempo; observar se há melhoria ou sinais de overfitting (val_loss crescendo).